In [0]:

from pyspark.sql.functions import col, sum, count, avg, max, min, year, month, dayofmonth, dayofweek, quarter, date_format, datediff, current_timestamp, lit, when, coalesce
from pyspark.sql.types import IntegerType, DoubleType, DateType, StringType

In [0]:

df_sales = spark.table("retailer.silver.sales")
df_products = spark.table("retailer.silver.products")
df_exchange = spark.table("retailer.silver.exchange_rates")

fact_sales = df_sales.join(
    df_products.select("product_key", "unit_price_usd"),
    "product_key",
    "left"
)


fact_sales = fact_sales.join(
    df_exchange.select(
        col("date").alias("exchange_date"),
        col("currency").alias("exchange_currency"),
        col("exchange").alias("exchange_rate")
    ),
    (fact_sales.order_date == col("exchange_date")) & 
    (fact_sales.currency_code == col("exchange_currency")),
    "left"
)


fact_sales = fact_sales \
    .withColumn("revenue_usd", 
                col("quantity") * col("unit_price_usd") * coalesce(col("exchange_rate"), lit(1.0))) \
   .withColumn("delivery_days", 
                when(
                    col("delivery_date").isNotNull() & (datediff(col("delivery_date"), col("order_date")) >= 0),
                    datediff(col("delivery_date"), col("order_date"))
                ).otherwise(None))

fact_sales = fact_sales.select(
    col("order_number"),
    col("line_item"),
    col("order_date"),
    col("delivery_date"),
    col("customer_key"),
    col("store_key"),
    col("product_key"),
    col("quantity"),
    col("unit_price_usd"),
    col("currency_code"),
    col("exchange_rate"),
    col("revenue_usd"),
    col("delivery_days")
)

fact_sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("retailer.gold.fact_sales")
     

df_sales = spark.table("retailer.silver.sales")

dim_date = df_sales.select(col("order_date").alias("date")).distinct() \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("day", dayofmonth(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("day_of_week", dayofweek(col("date"))) \
    .withColumn("month_name", date_format(col("date"), "MMMM")) \
    .withColumn("day_name", date_format(col("date"), "EEEE")) \
    .filter(col("date").isNotNull()) \
    .orderBy("date")

dim_date.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("retailer.gold.dim_date")
     

df_stores = spark.table("retailer.silver.stores")

dim_stores = df_stores.select(
    col("store_key"),
    col("country"),
    col("state"),
    col("square_meters"),
    col("open_date")
).withColumn("channel", lit("In-Store"))

dim_stores.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("retailer.gold.dim_stores")
     

df_products = spark.table("retailer.silver.products")

dim_products = df_products.select(
    col("product_key"),
    col("product_name"),
    col("brand"),
    col("color"),
    col("unit_cost_usd"),
    col("unit_price_usd"),
    col("subcategory_key"),
    col("subcategory"),
    col("category_key"),
    col("category")
)
dim_products.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("retailer.gold.dim_products")
     

df_customers = spark.table("retailer.silver.customers")

dim_customers = df_customers.select(
    col("customer_key"),
    col("gender"),
    col("name"),
    col("city"),
    col("state"),
    col("zip_code"),
    col("country"),
    col("continent"),
    col("birthday")
)

dim_customers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("retailer.gold.dim_customers")

In [0]:
from pyspark.sql.functions import round, when, lit, sum, count, avg, countDistinct

fact_sales = spark.table("retailer.gold.fact_sales")
dim_date = spark.table("retailer.gold.dim_date")
dim_customers = spark.table("retailer.gold.dim_customers")
dim_products = spark.table("retailer.gold.dim_products")
dim_stores = spark.table("retailer.gold.dim_stores")

data_cube = fact_sales \
    .join(dim_date, fact_sales.order_date == dim_date.date, "inner") \
    .join(dim_customers, fact_sales.customer_key == dim_customers.customer_key, "inner") \
    .join(dim_products, fact_sales.product_key == dim_products.product_key, "inner") \
    .join(dim_stores, fact_sales.store_key == dim_stores.store_key, "left") \
    .withColumn("channel", when(fact_sales.store_key.isNull(), lit("Online")).otherwise(lit("In-Store"))) \
    .groupBy(
        dim_date.year,
        dim_date.month,
        dim_date.month_name,
        dim_customers.country.alias("customer_country"),
        dim_customers.continent,
        dim_customers.gender,
        dim_products.category,
        dim_products.subcategory,
        dim_stores.country.alias("store_country"),
        "channel"
    ) \
    .agg(
        round(sum(fact_sales.revenue_usd), 2).alias("total_revenue"),
        sum(fact_sales.quantity).alias("total_quantity"),
        countDistinct(fact_sales.order_number).alias("order_count"),
        round(avg(fact_sales.revenue_usd), 2).alias("avg_order_value"),
        round(avg(fact_sales.delivery_days), 2).alias("avg_delivery_days"),
        sum(when(fact_sales.delivery_days.isNotNull(), 1).otherwise(0)).alias("delivered_order_count"),
        sum(when(fact_sales.delivery_days.isNull(), 1).otherwise(0)).alias("pending_order_count"),
        round(
            (sum(when(fact_sales.delivery_days.isNotNull(), 1).otherwise(0)) * 100.0) / countDistinct(fact_sales.order_number), 2
        ).alias("delivery_rate_pct")
    )

data_cube.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("retailer.gold.data_cube")